# TP 02 — Suivi des expériences avec MLflow

Ce notebook complète le TP 05 sans modifier son code. Il reprend le fichier Parquet, le prétraitement et la séparation temporelle du notebook 05, puis enregistre une nouvelle exécution et son modèle dans MLflow.

Les cellules de suivi peuvent être exécutées avec deux stockages différents : d'abord le stockage local par fichiers, puis un serveur MLflow local avec SQLite. Le code d'entraînement et d'enregistrement reste le même dans les deux cas.

## 1. Préparer l'environnement

Activez l'environnement `mlops` et ouvrez ce notebook depuis le dossier `notebooks`. MLflow doit être installé dans cet environnement. Si nécessaire, exécutez la cellule suivante puis redémarrez le noyau.

In [14]:
%pip install mlflow

Note: you may need to restart the kernel to use updated packages.


In [15]:
from pathlib import Path

import mlflow
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

current_dir = Path.cwd()
repo_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
data_path = repo_root / "data" / "yellow_tripdata_2026-05.parquet"
local_tracking_dir = repo_root / "mlruns" / "tp02-local"

if not data_path.exists():
    raise FileNotFoundError(f"Fichier Parquet introuvable : {data_path}")

print(f"Données : {data_path}")
print(f"Suivi local : {local_tracking_dir}")

Données : /workspaces/learning-mlops/data/yellow_tripdata_2026-05.parquet
Suivi local : /workspaces/learning-mlops/mlruns


## 2. Scénario 1 — stockage local

Aucun serveur n'est nécessaire : MLflow écrit les métadonnées et les artefacts dans `mlruns/tp02-local` à la racine du dépôt. Ce dossier dédié évite de mélanger ce TP avec d'anciens fichiers de suivi. L'interface pourra ensuite lire ce même dossier.

In [16]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

mlflow.set_tracking_uri(local_tracking_dir.as_uri())
mlflow.set_experiment("nyc-taxi-duration")

print(mlflow.get_tracking_uri())

file:///workspaces/learning-mlops/mlruns


## 3. Reprendre le protocole du TP 05

Le même fichier Parquet sert de source aux deux jeux. On échantillonne au plus 200 000 trajets, puis on trie les trajets par date et on utilise les premiers 80 % pour l'entraînement et les derniers 20 % pour la validation. Les chemins d'entraînement et de validation seront donc identiques ; le paramètre `validation_split` décrit la séparation réellement effectuée.

In [17]:
df = pd.read_parquet(data_path)
df["duration"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60
df = df[df["duration"].between(1, 60)].copy()

categorical = ["PULocationID", "DOLocationID"]
df[categorical] = df[categorical].astype(str)
df["PU_DO"] = df["PULocationID"] + "_" + df["DOLocationID"]

sample_size = min(200_000, len(df))
df = df.sample(n=sample_size, random_state=42)
df = df.sort_values("tpep_pickup_datetime").reset_index(drop=True)

split_index = int(len(df) * 0.8)
df_train = df.iloc[:split_index].copy()
df_val = df.iloc[split_index:].copy()

features = ["PU_DO", "trip_distance"]
train_dicts = df_train[features].to_dict(orient="records")
val_dicts = df_val[features].to_dict(orient="records")

dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
y_train = df_train["duration"].to_numpy()
y_val = df_val["duration"].to_numpy()

print(f"Taille entraînement : {len(df_train):,}")
print(f"Taille validation : {len(df_val):,}")

Taille entraînement : 160,000
Taille validation : 40,000


In [18]:
import numpy as np

# Lasso exige des indices 32 bits pour ces matrices creuses.
X_train = X_train.tocsc()
X_val = X_val.tocsc()
X_train.indices = X_train.indices.astype(np.int32)
X_train.indptr = X_train.indptr.astype(np.int32)
X_val.indices = X_val.indices.astype(np.int32)
X_val.indptr = X_val.indptr.astype(np.int32)

In [19]:
lasso_results = []

for alpha in [0.001, 0.01, 0.1]:
    model = Lasso(alpha=alpha, max_iter=10_000)
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)
    rmse = root_mean_squared_error(y_val, predictions)
    lasso_results.append({"model": "Lasso", "alpha": alpha, "rmse": rmse})
    print(f"Lasso alpha={alpha} | RMSE={rmse:.4f}")

xgb_model = XGBRegressor(
    max_depth=6,
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
)
xgb_model.fit(X_train, y_train)
xgb_predictions = xgb_model.predict(X_val)
xgb_rmse = root_mean_squared_error(y_val, xgb_predictions)
print(f"XGBoost max_depth=6 | RMSE={xgb_rmse:.4f}")

lasso_results.append({"model": "XGBoost", "alpha": None, "rmse": xgb_rmse})
results_df = pd.DataFrame(lasso_results).sort_values("rmse")
display(results_df)

best_lasso = min(lasso_results, key=lambda result: result["rmse"])
if best_lasso["rmse"] < xgb_rmse:
    selected_name = "Lasso"
    selected_rmse = best_lasso["rmse"]
    selected_model = Lasso(alpha=best_lasso["alpha"], max_iter=10_000)
    selected_model.fit(X_train, y_train)
else:
    selected_name = "XGBoost"
    selected_rmse = xgb_rmse
    selected_model = xgb_model

print("Modèle retenu :", selected_name)
print("RMSE :", selected_rmse)

,model,validation_rmse
2,XGBoost,4.351970
0,LinearRegression,4.853419
1,Lasso,8.780913


Modèle retenu : XGBoost | RMSE : 4.3520


## 4. Enregistrer l'exécution

Le modèle est emballé avec le `DictVectorizer` déjà ajusté. L'artefact MLflow accepte ainsi directement les dictionnaires de caractéristiques du TP 05 et conserve l'ordre de transformation appris à l'entraînement.

In [41]:
tracking_model = Pipeline(
    [
        ("vectorizer", dv),
        ("regressor", selected_model),
    ]
)

if selected_name == "Lasso":
    selected_model_params = {
        "alpha": selected_model.alpha,
        "max_iter": selected_model.max_iter,
    }
else:
    model_params = selected_model.get_params(deep=False)
    selected_model_params = {
        name: model_params[name]
        for name in ("max_depth", "n_estimators", "learning_rate", "random_state", "objective")
    }

skops_trusted_types = None
if selected_name == "XGBoost":
    skops_trusted_types = [
        "xgboost.core.Booster",
        "xgboost.sklearn.XGBRegressor",
    ]

def log_training_run():
    with mlflow.start_run() as active_run:
        mlflow.set_tag("task", "taxi-duration-prediction")
        mlflow.log_params(
            {
                "model": selected_name,
                "sample_size": sample_size,
                "validation_ratio": 0.2,
                "train_data_path": str(data_path.resolve()),
                "validation_data_path": str(data_path.resolve()),
                "validation_split": "sampled_then_sorted_by_pickup_time_first_80_percent",
                **{
                    f"model_param_{name}": value
                    for name, value in selected_model_params.items()
                },
            }
        )
        mlflow.log_metric("validation_rmse", float(selected_rmse))
        mlflow.sklearn.log_model(
            sk_model=tracking_model,
            name="model",
            skops_trusted_types=skops_trusted_types,
        )

    return active_run.info.run_id

In [45]:
run_id_local = log_training_run()
print(f"Exécution locale enregistrée : {run_id_local}")

🏃 View run delicate-skink-151 at: http://127.0.0.1:5000/#/experiments/1/runs/4512bedefeb94b93a1c17ef9416ed94f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Exécution locale enregistrée : 4512bedefeb94b93a1c17ef9416ed94f


Depuis un terminal ouvert à la racine du dépôt, affichez l'interface du stockage local :

```bash
MLFLOW_ALLOW_FILE_STORE=true mlflow ui --host 0.0.0.0 --port 5000 --backend-store-uri ./mlruns/tp02-local
```

Ouvrez `http://127.0.0.1:5000` (ou le port 5000 dans Codespaces) et vérifiez l'expérience `nyc-taxi-duration`, les paramètres, la RMSE et l'artefact `model`.

## 5. Scénario 2 — serveur local et SQLite

Arrêtez l'interface précédente. Depuis un terminal à la racine du dépôt, démarrez le serveur MLflow et laissez-le actif :

```bash
mlflow server \
  --host 0.0.0.0 \
  --port 5000 \
  --backend-store-uri sqlite:///mlflow.db \
  --serve-artifacts \
  --artifacts-destination ./mlartifacts
```

Le client notebook envoie maintenant les métadonnées et les artefacts au serveur. Exécutez la cellule de configuration, puis celle qui appelle `log_training_run()` pour enregistrer une exécution dans SQLite.

In [43]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("nyc-taxi-duration")

print(mlflow.get_tracking_uri())

http://127.0.0.1:5000


In [44]:
run_id_server = log_training_run()
print(f"Exécution du serveur enregistrée : {run_id_server}")

🏃 View run worried-toad-256 at: http://127.0.0.1:5000/#/experiments/1/runs/b63d9305897d4be9a91e02df4a0e3a87
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Exécution du serveur enregistrée : b63d9305897d4be9a91e02df4a0e3a87


Dans l'interface servie sur le port 5000, retrouvez l'exécution enregistrée sur le serveur et vérifiez ses chemins Parquet, paramètres, métrique et artefact `model`. Le fichier `mlflow.db` contient les métadonnées ; les artefacts sont conservés dans `mlartifacts`.

## 6. Retrouver un run et recharger son modèle

Dans l'interface MLflow, choisissez un run qui contient déjà l'artefact `model` et copiez son identifiant. La valeur par défaut ci-dessous correspond au run serveur créé dans ce notebook ; remplacez-la si vous souhaitez charger un autre run existant. Cette étape ne lance aucun nouvel entraînement.

In [46]:
selected_run_id = run_id_server
selected_run = mlflow.get_run(selected_run_id)
model_uri = f"runs:/{selected_run_id}/model"
reloaded_model = mlflow.sklearn.load_model(model_uri)

example = [{"PU_DO": "161_236", "trip_distance": 3.2}]
prediction = float(reloaded_model.predict(example)[0])

print(f"Run sélectionné : {selected_run.info.run_id}")
print(f"Paramètres : {selected_run.data.params}")
print(f"Métriques : {selected_run.data.metrics}")
print(f"URI du modèle rechargé : {model_uri}")
print(f"Prédiction de durée : {prediction:.2f} minutes")

Run sélectionné : b63d9305897d4be9a91e02df4a0e3a87
Paramètres : {'model': 'XGBoost', 'sample_size': '200000', 'validation_ratio': '0.2', 'train_data_path': '/workspaces/learning-mlops/data/yellow_tripdata_2026-05.parquet', 'validation_data_path': '/workspaces/learning-mlops/data/yellow_tripdata_2026-05.parquet', 'validation_split': 'sampled_then_sorted_by_pickup_time_first_80_percent', 'model_param_max_depth': '6', 'model_param_n_estimators': '100', 'model_param_learning_rate': '0.1', 'model_param_random_state': '42', 'model_param_objective': 'reg:squarederror'}
Métriques : {'validation_rmse': 4.351969735010792}
URI du modèle rechargé : runs:/b63d9305897d4be9a91e02df4a0e3a87/model
Prédiction de durée : 14.94 minutes


## 7. Vérification

1. Le scénario 1 conserve les métadonnées et les artefacts dans le dossier local `mlruns` ; il ne nécessite pas de serveur.
2. Le scénario 2 enregistre les métadonnées dans SQLite (`mlflow.db`) et les artefacts dans `mlartifacts`, via un serveur MLflow local.
3. SQLite fournit un stockage structuré des expériences, des runs, des paramètres et des métriques.
4. `log_training_run()` reste identique : c'est l'URI configurée avant le run qui détermine le stockage utilisé.
5. Le scénario distant n'est pas déployé : il nécessiterait notamment une infrastructure partagée, une base distante, un stockage objet et une gestion sécurisée des accès.